In [1]:
!pip install --quiet langchain-community langchain openai tiktoken faiss-cpu
!pip install --quiet sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 489.1/489.1 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.1/157.1 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.7/343.7 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.3/211.3 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 100.1 MB/s eta 0:00:0000:010:01
   ━━━━

In [2]:
import os

# Where to clone
REPO_URL = "https://github.com/ratulsaha2021/Fine-Tuning-LLaMA-3.1-8B-for-Bengali-Empathetic-Dialogue"
REPO_DIR = "/kaggle/working/repo_bengali_llama"

# Clone if not already
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo already cloned:", REPO_DIR)

# List files to confirm
for root, dirs, files in os.walk(REPO_DIR):
    print(root)
    for f in files:
        print("  ", f)
    break  # only top level


Cloning into '/kaggle/working/repo_bengali_llama'...
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 13 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (13/13), 5.88 MiB | 24.01 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/kaggle/working/repo_bengali_llama
   README.md
   llama-3-1-8b-instruct-fine-tuning-on-bengali-empat.ipynb
   Fine-Tuning LLaMA 3.1-8B for Bengali Empathetic Dialogue.docx


In [3]:
import os

REPO_DIR = "/kaggle/working/repo_bengali_llama"

def load_repo_texts(repo_dir):
    texts = []
    exts = {".md", ".txt", ".py", ".ipynb"}  # we will handle .ipynb later if needed

    for root, dirs, files in os.walk(repo_dir):
        for fname in files:
            path = os.path.join(root, fname)
            _, ext = os.path.splitext(fname)

            # For now, only simple text-type files
            if ext.lower() not in exts:
                continue

            if ext.lower() == ".ipynb":
                # skip notebooks for now to keep it simple
                continue

            try:
                with open(path, "r", encoding="utf-8") as f:
                    text = f.read()
                texts.append({"path": path, "text": text})
            except Exception as e:
                print("Could not read:", path, "Error:", e)

    return texts

docs = load_repo_texts(REPO_DIR)
len(docs), [d["path"] for d in docs]


(1, ['/kaggle/working/repo_bengali_llama/README.md'])

In [4]:
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.document import Document

# 1) Prepare text chunks from docs
def make_chunks(docs, chunk_size=600, chunk_overlap=100):
    all_chunks = []
    for d in docs:
        text = d["text"]
        path = d["path"]
        start = 0
        while start < len(text):
            end = start + chunk_size
            chunk = text[start:end]
            all_chunks.append(
                Document(page_content=chunk, metadata={"source": path})
            )
            start += chunk_size - chunk_overlap
    return all_chunks

chunks = make_chunks(docs)
len(chunks), chunks[0].metadata, chunks[0].page_content[:200]


2026-01-06 16:08:29.732908: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767715709.889129      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767715709.933943      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

(4,
 {'source': '/kaggle/working/repo_bengali_llama/README.md'},
 '# Bengali-Empathetic-LLaMA-3.1-FineTuning\n\n# Fine-tuning LLaMA 3.1-8B-Instruct to generate empathetic responses in Bengali using Parameter-Efficient Fine-Tuning (PEFT) and the Unsloth framework.\n\n🚀 Pr')

In [5]:
# 2) Load embedding model (small, good for Kaggle)
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

class STEmbeddings:
    def __init__(self, model):
        self.model = model
    def embed_documents(self, texts):
        return self.model.encode(texts, show_progress_bar=False).tolist()
    def embed_query(self, text):
        return self.model.encode([text], show_progress_bar=False)[0].tolist()

embeddings = STEmbeddings(embed_model)

# 3) Create FAISS vector store
vector_store = FAISS.from_documents(chunks, embeddings)

# 4) Quick test: retrieve top 3 chunks for a sample query
query = "What is this project about?"
docs_retrieved = vector_store.similarity_search(query, k=3)

len(docs_retrieved), docs_retrieved[0].metadata, docs_retrieved[0].page_content[:300]


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

TypeError: 'STEmbeddings' object is not callable

In [6]:
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.document import Document
from langchain.embeddings.base import Embeddings  # <-- important

# 1) Prepare text chunks from docs
def make_chunks(docs, chunk_size=600, chunk_overlap=100):
    all_chunks = []
    for d in docs:
        text = d["text"]
        path = d["path"]
        start = 0
        while start < len(text):
            end = start + chunk_size
            chunk = text[start:end]
            all_chunks.append(
                Document(page_content=chunk, metadata={"source": path})
            )
            start += chunk_size - chunk_overlap
    return all_chunks

chunks = make_chunks(docs)
print("Num chunks:", len(chunks))
print("First chunk meta:", chunks[0].metadata)
print("First chunk preview:", chunks[0].page_content[:200])

# 2) Load embedding model (small, good for Kaggle)
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# 3) Proper Embeddings subclass for LangChain
class STEmbeddings(Embeddings):
    def __init__(self, model):
        self.model = model

    def embed_documents(self, texts):
        return self.model.encode(texts, show_progress_bar=False).tolist()

    def embed_query(self, text):
        return self.model.encode([text], show_progress_bar=False)[0].tolist()

embeddings = STEmbeddings(embed_model)

# 4) Create FAISS vector store
vector_store = FAISS.from_documents(chunks, embeddings)

# 5) Quick test: retrieve top 3 chunks for a sample query
query = "What is this project about?"
docs_retrieved = vector_store.similarity_search(query, k=3)

len(docs_retrieved), docs_retrieved[0].metadata, docs_retrieved[0].page_content[:300]


Num chunks: 4
First chunk meta: {'source': '/kaggle/working/repo_bengali_llama/README.md'}
First chunk preview: # Bengali-Empathetic-LLaMA-3.1-FineTuning

# Fine-tuning LLaMA 3.1-8B-Instruct to generate empathetic responses in Bengali using Parameter-Efficient Fine-Tuning (PEFT) and the Unsloth framework.

🚀 Pr


(3,
 {'source': '/kaggle/working/repo_bengali_llama/README.md'},
 ' and culturally/emotionally aware dialogue.\n\n🛠️ Technical Stack & Design\n\nBase Model: LLaMA 3.1-8B-Instruct\n\nQuantization: 4-bit NormalFloat (NF4)\n\nFine-Tuning Method: LoRA (Low-Rank Adaptation)\n\nFramework: Unsloth (2x faster training, 70% less memory)\n\nDesign Pattern: Strategy Pattern for modular T')

In [7]:
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.document import Document
from langchain.embeddings.base import Embeddings

# 1) Prepare text chunks from docs
def make_chunks(docs, chunk_size=600, chunk_overlap=100):
    all_chunks = []
    for d in docs:
        text = d["text"]
        path = d["path"]
        start = 0
        while start < len(text):
            end = start + chunk_size
            chunk = text[start:end]
            all_chunks.append(
                Document(page_content=chunk, metadata={"source": path})
            )
            start += chunk_size - chunk_overlap
    return all_chunks

chunks = make_chunks(docs)
print("Num chunks:", len(chunks))
print("First chunk meta:", chunks[0].metadata)
print("First chunk preview:", chunks[0].page_content[:200])

# 2) Load embedding model (small, good for Kaggle)
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# 3) Proper Embeddings subclass for LangChain
class STEmbeddings(Embeddings):
    def __init__(self, model):
        self.model = model

    def embed_documents(self, texts):
        return self.model.encode(texts, show_progress_bar=False).tolist()

    def embed_query(self, text):
        return self.model.encode([text], show_progress_bar=False)[0].tolist()

embeddings = STEmbeddings(embed_model)

# 4) Create FAISS vector store
vector_store = FAISS.from_documents(chunks, embeddings)

# 5) Quick test: retrieve top 3 chunks for a sample query
query = "What is this project about?"
docs_retrieved = vector_store.similarity_search(query, k=3)

len(docs_retrieved), docs_retrieved[0].metadata, docs_retrieved[0].page_content[:300]


Num chunks: 4
First chunk meta: {'source': '/kaggle/working/repo_bengali_llama/README.md'}
First chunk preview: # Bengali-Empathetic-LLaMA-3.1-FineTuning

# Fine-tuning LLaMA 3.1-8B-Instruct to generate empathetic responses in Bengali using Parameter-Efficient Fine-Tuning (PEFT) and the Unsloth framework.

🚀 Pr


(3,
 {'source': '/kaggle/working/repo_bengali_llama/README.md'},
 ' and culturally/emotionally aware dialogue.\n\n🛠️ Technical Stack & Design\n\nBase Model: LLaMA 3.1-8B-Instruct\n\nQuantization: 4-bit NormalFloat (NF4)\n\nFine-Tuning Method: LoRA (Low-Rank Adaptation)\n\nFramework: Unsloth (2x faster training, 70% less memory)\n\nDesign Pattern: Strategy Pattern for modular T')

In [8]:
import textwrap

def answer_query(query, k=3, max_chars=700):
    # 1) Retrieve similar chunks
    retrieved = vector_store.similarity_search(query, k=k)

    # 2) Concatenate them as context
    context_parts = []
    for doc in retrieved:
        context_parts.append(doc.page_content)
    context = "\n\n".join(context_parts)
    context = context[:max_chars]

    # 3) Very simple rule-based "answer"
    answer = f"Question: {query}\n\n"
    answer += "Answer (from README context):\n"
    answer += textwrap.shorten(context.replace("\n", " "), width=max_chars, placeholder="...")

    return answer

# Test calls
print(answer_query("What is this project about?", k=3, max_chars=600))
print("-----")
print(answer_query("Which base model is used?", k=3, max_chars=600))
print("-----")
print(answer_query("What metrics did the fine-tuned model achieve?", k=3, max_chars=600))


Question: What is this project about?

Answer (from README context):
and culturally/emotionally aware dialogue. 🛠️ Technical Stack & Design Base Model: LLaMA 3.1-8B-Instruct Quantization: 4-bit NormalFloat (NF4) Fine-Tuning Method: LoRA (Low-Rank Adaptation) Framework: Unsloth (2x faster training, 70% less memory) Design Pattern: Strategy Pattern for modular Tuner, Processor, and Evaluator classes. 📊 Key Results The model was fine-tuned on a Bengali Empathetic Dialogue corpus and achieved the following metrics: Metric Result BLEU Score 0.7267 ROUGE-L 1.0000 Average NLL 4.8774 Perplexity 131.28 🧠 Theoretical Background In adherence to modern
-----
Question: Which base model is used?

Answer (from README context):
and culturally/emotionally aware dialogue. 🛠️ Technical Stack & Design Base Model: LLaMA 3.1-8B-Instruct Quantization: 4-bit NormalFloat (NF4) Fine-Tuning Method: LoRA (Low-Rank Adaptation) Framework: Unsloth (2x faster training, 70% less memory) Design Pattern: Strategy Patter

In [9]:
import textwrap

def answer_query_clean(query, k=3, max_chars=500):
    # 1) Retrieve similar chunks
    retrieved = vector_store.similarity_search(query, k=k)

    # 2) Take the best chunk only (most relevant)
    best = retrieved[0].page_content

    # 3) Try to keep only a few lines around the most relevant part
    #    For simplicity, split into sentences and keep the first 5.
    text = best.replace("\n", " ")
    sentences = [s.strip() for s in text.split(".") if s.strip()]
    short = ". ".join(sentences[:5]) + "."

    # 4) Clip to max_chars
    short = textwrap.shorten(short, width=max_chars, placeholder="...")

    # 5) Build final answer string
    ans = f"Question: {query}\n\n"
    ans += "Answer:\n"
    ans += short
    return ans

# Test it
print(answer_query_clean("What is this project about?", k=3, max_chars=400))
print("-----")
print(answer_query_clean("Which base model is used?", k=3, max_chars=400))
print("-----")
print(answer_query_clean("What metrics did the fine-tuned model achieve?", k=3, max_chars=400))


Question: What is this project about?

Answer:
and culturally/emotionally aware dialogue. 🛠️ Technical Stack & Design Base Model: LLaMA 3. 1-8B-Instruct Quantization: 4-bit NormalFloat (NF4) Fine-Tuning Method: LoRA (Low-Rank Adaptation) Framework: Unsloth (2x faster training, 70% less memory) Design Pattern: Strategy Pattern for modular Tuner, Processor, and Evaluator classes. 📊 Key Results The model was fine-tuned on a Bengali Empathetic...
-----
Question: Which base model is used?

Answer:
and culturally/emotionally aware dialogue. 🛠️ Technical Stack & Design Base Model: LLaMA 3. 1-8B-Instruct Quantization: 4-bit NormalFloat (NF4) Fine-Tuning Method: LoRA (Low-Rank Adaptation) Framework: Unsloth (2x faster training, 70% less memory) Design Pattern: Strategy Pattern for modular Tuner, Processor, and Evaluator classes. 📊 Key Results The model was fine-tuned on a Bengali Empathetic...
-----
Question: What metrics did the fine-tuned model achieve?

Answer:
and culturally/emotionally awa

In [1]:
%%writefile app.py
import streamlit as st
import textwrap
import pickle
import os

from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.document import Document
from langchain.embeddings.base import Embeddings

# ---------- Load / build vector store ----------

REPO_DIR = "/kaggle/working/repo_bengali_llama"

def load_repo_texts(repo_dir):
    texts = []
    exts = {".md", ".txt"}
    for root, dirs, files in os.walk(repo_dir):
        for fname in files:
            path = os.path.join(root, fname)
            _, ext = os.path.splitext(fname)
            if ext.lower() not in exts:
                continue
            try:
                with open(path, "r", encoding="utf-8") as f:
                    text = f.read()
                texts.append({"path": path, "text": text})
            except Exception:
                pass
    return texts

def make_chunks(docs, chunk_size=600, chunk_overlap=100):
    all_chunks = []
    for d in docs:
        text = d["text"]
        path = d["path"]
        start = 0
        while start < len(text):
            end = start + chunk_size
            chunk = text[start:end]
            all_chunks.append(
                Document(page_content=chunk, metadata={"source": path})
            )
            start += chunk_size - chunk_overlap
    return all_chunks

class STEmbeddings(Embeddings):
    def __init__(self, model):
        self.model = model
    def embed_documents(self, texts):
        return self.model.encode(texts, show_progress_bar=False).tolist()
    def embed_query(self, text):
        return self.model.encode([text], show_progress_bar=False)[0].tolist()

@st.cache_resource
def get_vector_store():
    docs = load_repo_texts(REPO_DIR)
    chunks = make_chunks(docs)
    embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    embeddings = STEmbeddings(embed_model)
    vs = FAISS.from_documents(chunks, embeddings)
    return vs

vector_store = get_vector_store()

# ---------- Simple QA function ----------

def answer_query_clean(query, k=3, max_chars=500):
    retrieved = vector_store.similarity_search(query, k=k)
    best = retrieved[0].page_content
    text = best.replace("\n", " ")
    sentences = [s.strip() for s in text.split(".") if s.strip()]
    short = ". ".join(sentences[:5]) + "."
    short = textwrap.shorten(short, width=max_chars, placeholder="...")
    ans = short
    return ans

# ---------- Streamlit UI ----------

st.set_page_config(page_title="Bengali LLaMA Repo Chatbot")
st.title("📁 Bengali LLaMA Repo Chatbot")
st.write("Ask questions about the GitHub project: **Fine-Tuning LLaMA 3.1-8B for Bengali Empathetic Dialogue**.")

if "messages" not in st.session_state:
    st.session_state.messages = []

# Show history
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

user_input = st.chat_input("Ask something about the repo...")

if user_input:
    # Add user message
    st.session_state.messages.append({"role": "user", "content": user_input})
    with st.chat_message("user"):
        st.write(user_input)

    # Bot response
    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            answer = answer_query_clean(user_input, k=3, max_chars=500)
            st.write(answer)
    st.session_state.messages.append({"role": "assistant", "content": answer})


Writing app.py


In [2]:
!pip install --quiet streamlit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 66.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 117.8 MB/s eta 0:00:0000:01


In [ ]:
!streamlit run app.py --server.port 8501 --server.headless true





  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.19.2.2:8501
  External URL: http://34.45.109.170:8501

